# Documentation: Generating a Chess Position Evaluation Dataset

Data source: Lichess, June 2025 (~93M games).
Goal: build a balanced set of positions with engine evaluations without exhausting memory or CPU.

## 1. Scope and assumptions

We aim to prepare a large dataset (e.g. 1M positions) containing:
- Position FEN
- Stockfish evaluation (cp + normalized + mate flag)
- Game phase (OPEN / MID / END)
- Basic material features
- Technical metadata (evaluation time, depth)

Constraints:
- Do not load whole PGN into memory
- Keep engine cost low
- Maintain phase balance
- Avoid trivial duplicates

---

## 2. Problem: Data scale

Input: ~93M games → not feasible to:
- Load entire file at once
- Fully decompress the .zst archive to disk

Solution:
- Streaming PGN parsing (`chess.pgn.read_game(stream)`)
- `.zst` handling via:
  - Python `zstandard` module (preferred)
  - Fallback external process `zstd -dc`

---

## 3. Problem: Engine evaluation cost

Evaluating every position is too expensive.

Solutions:
- Sampling (only selected positions)
- Batch evaluation + parallel processes
- Very short time per position (fast eval, e.g. 12 ms)
- Clamp score range to [-2000, 2000]

---

## 4. Problem: Selecting positions (sampling)

Aim: informative and tactical positions without overloading the dataset with opening moves.

Rules:
- Consider a position if:
  - `ply % 4 == 0` (or %2 in endgame focus mode), OR
  - the move is a capture, OR
  - the move is a promotion
- Limit per game: `MAX_POSITIONS_PER_GAME` (e.g. 8)

Reasoning:
- Prevent domination by long games
- Capture tactical/material transitions

---

## 5. Problem: Game phase identification

Move number alone is unreliable.

Heuristic:
- `phase_raw = 4*Q + 2*R + (B + N)`; normalize by 24
- END if:
  - no queens, OR
  - total pieces+pawns ≤ 10 (excluding kings), OR
  - no heavy pieces and ≤ 1 minor piece
- Else: OPEN if `phase_norm ≥ 0.66`, otherwise MID

Purpose: balancing dataset according to configured targets.

---

## 6. Problem: Endgame under-representation

Endgames are naturally rare.

Mechanism:
- Planned share = `target_end / total_target`
- If `(planned - current) > ENDGAME_FOCUS_TRIGGER_GAP` → enable endgame focus
- In focus: denser sampling (every 2 ply)
- Auto disable once caught up

---

## 7. Problem: Duplicate positions

Many repeated structures (openings, technical endings).

Strategy:
- Reduced FEN: first 4 fields (piece placement, side to move, castling rights, en passant square)
- Ignore halfmove clock and fullmove number
- Maintain `seen_fen` set

Rationale:
- Avoid artificial variety from counters only
- Reduce over-representation of identical early structures

---

## 8. Problem: Throughput and coordination

Need efficient flow:
- Collect metadata → `pending`
- When batch full → parallel evaluation → results go to shard buffer
- When shard full → write to CSV

Benefits:
- Lower evaluation overhead
- Crash resilience (only lose unwritten buffer)

---

## 9. Problem: Score normalization

Raw centipawn scores can be extreme.

Solution:
- Clamp to [-2000, 2000]
- `eval_norm = eval_cp / 2000`
- Mates mapped to ±2000 + separate `mate_flag`

Purpose: stable numeric range for downstream models.

---

## 10. Problem: Single giant output file

Issues:
- Risky (corruption, interruption)
- Hard to transfer
- Heavy to open

Solution:
- Sharding (e.g. every 100k positions)
- Files named `positions_shard_0001.csv`, etc.
- Flush remainder on finalize

---

## 11. Problem: Errors and interruptions

Potential failures:
- Corrupt PGN segments
- Engine exceptions
- User interruption (Ctrl+C)
- Missing decompression tooling

Mitigations:
- `try/except` around PGN parsing (log + continue)
- Engine failure → neutral eval (0 cp, depth -1)
- SIGINT handler calls `finalize()`
- Check for `zstandard` / `zstd` availability

---

## 12. Problem: Parameter flexibility

Different environments → different constraints.

Configurable via CLI:
- Phase targets (`--target-open`, etc.) or `--target-total` (30/30/40 split)
- Batch size
- Engine instances
- Time per position
- Hash size
- Disable endgame focus
- Limit number of games (`--max-games`) for testing

Trade-offs:
- Too many instances > CPU cores → slowdown
- Too small batch → overhead; too large → slower feedback

---

## 13. Problem: Preserving result order

Parallel workers return results out of order.

Solution:
- Attach index to each task
- Reconstruct ordered list by index slot assignment

---

## 14. Problem: Extendability

Future needs:
- Higher-depth “gold subset”
- Additional features (pawn structure hash, mobility)
- Multi-month aggregation

Design kept modular to allow this.

---

## 15. Potential extensions (roadmap)

- Gold subset with deeper time / MultiPV
- Parquet export
- Sampling based on eval deltas (pre vs post move)
- Statistics for sampling reasons (capture vs interval)
- Position feature engineering (king safety, pawn islands)
- Distributed multi-node processing

---

## 16. Post-generation validation

Suggested checks:
- Target counts match (OPEN/MID/END)
- Endgame focus activation visible in logs when lagging
- Zero (or negligible) duplicates after reduced FEN
- Distribution of `eval_cp` (no overwhelming extreme tails)
- Manual inspection of random END positions

---

## 17. Flow overview

```
stream PGN -> parse game -> iterate moves
  -> sampling decision -> classify phase -> deduplicate
    -> build PositionMeta -> batch threshold?
       -> parallel engine eval -> EvalResult list
          -> append to shard buffer -> shard full? write CSV
finalize: flush pending, write remaining shard
```

---

## 18. Example run (test scale)

```
python prepare_dataset.py \
  --pgn lichess_db_standard_rated_2025-06.pgn.zst \
  --stockfish /usr/bin/stockfish \
  --out-dir data_shards \
  --target-total 50000 \
  --batch-eval-size 400 \
  --engine-instances 6 \
  --engine-time 0.010
```

---

## 19. Summary comparison table

| Problem | Technique | Reason |
|---------|----------|--------|
| Scale | Streaming | Low memory usage |
| Rare endgames | Adaptive focus | Balanced phase distribution |
| Engine cost | Fast eval + parallel | Throughput |
| Duplicates | Reduced FEN | Genuine variety |
| Interruption risk | Sharding + finalize | Minimal loss |
| Phase balance | Targets + monitoring | Controlled composition |

---

## 20. Summary

This solution addresses:
- Large-scale ingestion (streaming)
- Efficient evaluation (batch + multiprocessing)
- Phase balance (targets + adaptive endgame focus)
- Diversity (sampling rules + deduplication)
- Stability (error handling, signal handling)
- Practical usability (sharded CSV, normalized scores)

Prepared in a way that supports iterative improvement without a rewrite.

End of documentation.